In [21]:
#load the necessary libraries

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit  #edited by claude code


In [22]:
#load the clinical and methylation datasets; both are obtained from GDC TCGA KIRC cohort from Xenahub
clinical_kirc_pd = pd.read_csv('../data/others/kirc/TCGA-KIRC.clinical.tsv', sep='\t')
methylation_kirc_pd = pd.read_csv('../data/others/kirc/TCGA-KIRC.methylation450.tsv', sep='\t')

In [23]:
#view the first few rows of the clinical and methylation datasets
print("This is what the clinical data looks like:")
print(clinical_kirc_pd)
print("------------------------------------------------------")
print("\n This is what the methylation data looks like:")
print(methylation_kirc_pd)

This is what the clinical data looks like:
               sample                                    id  \
0    TCGA-B0-5695-01A  fb9bafa5-7133-4955-8156-4eb6763dc8e1   
1    TCGA-BP-4807-01A  33cf0893-0411-4758-aa90-602bfedf0850   
2    TCGA-BP-4807-11A  33cf0893-0411-4758-aa90-602bfedf0850   
3    TCGA-BP-4995-01A  9245a557-01ea-4446-b9e9-313f6ab18834   
4    TCGA-BP-4995-11A  9245a557-01ea-4446-b9e9-313f6ab18834   
..                ...                                   ...   
945  TCGA-B0-5402-11A  d7ab7ec0-3de7-4ffd-a5ac-f75579355b2a   
946  TCGA-B0-4821-01A  8d769b38-862f-46e3-a426-d09f79f28de6   
947  TCGA-B0-4821-11A  8d769b38-862f-46e3-a426-d09f79f28de6   
948  TCGA-CZ-4866-01A  291f45ab-1b77-456a-9783-06106fbe3053   
949  TCGA-CZ-4866-11A  291f45ab-1b77-456a-9783-06106fbe3053   

                     disease_type                               case_id  \
0    Adenomas and Adenocarcinomas  fb9bafa5-7133-4955-8156-4eb6763dc8e1   
1    Adenomas and Adenocarcinomas  33cf0893-0411-4

In [24]:
#transpose the methylation dataframe as the current format has samples as columns and features as rows
methylation_kirc_pd.set_index('Composite Element REF', inplace=True)
methylation_kirc_pd_old = methylation_kirc_pd.transpose()

print("\n This is what the transposed methylation data looks like:")
print(methylation_kirc_pd_old)


 This is what the transposed methylation data looks like:
Composite Element REF  cg00000029  cg00000108  cg00000109  cg00000165  \
TCGA-B4-5834-01A         0.614927    0.972071    0.612456    0.188422   
TCGA-DV-5575-01A         0.321158    0.964997    0.758764    0.130063   
TCGA-B0-5108-01A         0.450301    0.972311    0.678896    0.197717   
TCGA-B0-5703-01A         0.816181    0.964033    0.723863    0.152583   
TCGA-CJ-4905-11A         0.387471    0.960496    0.860965    0.203964   
...                           ...         ...         ...         ...   
TCGA-BP-4993-01A         0.461124    0.965083    0.599614    0.163763   
TCGA-CZ-4865-01A         0.594906    0.969403    0.687554    0.343633   
TCGA-BP-4782-11A         0.488778    0.959676    0.891868    0.185912   
TCGA-B0-4945-11A         0.528901    0.970876    0.923883    0.124858   
TCGA-B4-5843-01A         0.490657    0.969888    0.519000    0.104241   

Composite Element REF  cg00000236  cg00000289  cg00000292  cg000

In [25]:
#drop the probes (columns) that have more than 1% missing values across all samples (rows)
#(e.g., if you have 370 tumor samples, keep probes that have at least 366 valid values)
threshold = int(methylation_kirc_pd_old.shape[0] * 0.99)

methylation_kirc_pd = methylation_kirc_pd_old.dropna(axis=1, thresh=threshold)

print(f"Original probes: {methylation_kirc_pd_old.shape[1]}")
print(f"Probes after dropping bad ones: {methylation_kirc_pd.shape[1]}")
print("This is what the methylation data looks like after dropping probes with more than 1% missing values:")
print(methylation_kirc_pd)



Original probes: 486427
Probes after dropping bad ones: 371082
This is what the methylation data looks like after dropping probes with more than 1% missing values:
Composite Element REF  cg00000108  cg00000236  cg00000292  cg00000321  \
TCGA-B4-5834-01A         0.972071    0.917789    0.426973    0.270070   
TCGA-DV-5575-01A         0.964997    0.931517    0.492524    0.310941   
TCGA-B0-5108-01A         0.972311    0.909915    0.601544    0.283628   
TCGA-B0-5703-01A         0.964033    0.874732    0.590178    0.454810   
TCGA-CJ-4905-11A         0.960496    0.907699    0.489555    0.437536   
...                           ...         ...         ...         ...   
TCGA-BP-4993-01A         0.965083    0.926505    0.766406    0.518425   
TCGA-CZ-4865-01A         0.969403    0.915849    0.515882    0.515539   
TCGA-BP-4782-11A         0.959676    0.936125    0.496571    0.396565   
TCGA-B0-4945-11A         0.970876    0.924377    0.491113    0.455013   
TCGA-B4-5843-01A         0.969888

In [26]:
#view the the clinical features (columns) in the clinical dataset
print("These are the clinical features (columns) in the clinical dataset:")
print(clinical_kirc_pd.columns)

These are the clinical features (columns) in the clinical dataset:
Index(['sample', 'id', 'disease_type', 'case_id', 'submitter_id',
       'primary_site', 'cigarettes_per_day.exposures',
       'alcohol_history.exposures', 'years_smoked.exposures',
       'race.demographic', 'gender.demographic', 'ethnicity.demographic',
       'vital_status.demographic', 'age_at_index.demographic',
       'days_to_birth.demographic', 'year_of_birth.demographic',
       'year_of_death.demographic', 'primary_site.project',
       'project_id.project', 'disease_type.project', 'name.project',
       'name.program.project', 'tissue_source_site_id.tissue_source_site',
       'code.tissue_source_site', 'name.tissue_source_site',
       'project.tissue_source_site', 'bcr_id.tissue_source_site',
       'days_to_death.demographic', 'pack_years_smoked.exposures',
       'entity_submitter_id.annotations', 'notes.annotations',
       'submitter_id.annotations', 'classification.annotations',
       'entity_id.anno

In [27]:
#retain only the relevant clinical features for our downstream analysis

columns_to_keep = [
    'sample',                                  #keep as index to match methylation matrix
    'tissue_type.samples'                    #normal vs tumor
]

#filter the clinical dataset to keep only the relevant columns
clinical_kirc_all = clinical_kirc_pd[columns_to_keep]

print("This is our dataset for our use-case:")
print(clinical_kirc_all)


This is our dataset for our use-case:
               sample tissue_type.samples
0    TCGA-B0-5695-01A               Tumor
1    TCGA-BP-4807-01A               Tumor
2    TCGA-BP-4807-11A              Normal
3    TCGA-BP-4995-01A               Tumor
4    TCGA-BP-4995-11A              Normal
..                ...                 ...
945  TCGA-B0-5402-11A              Normal
946  TCGA-B0-4821-01A               Tumor
947  TCGA-B0-4821-11A              Normal
948  TCGA-CZ-4866-01A               Tumor
949  TCGA-CZ-4866-11A              Normal

[950 rows x 2 columns]


In [28]:
#extract the sample names from the methylation dataset to filter the clinical dataset to keep only the samples that are present in the methylation dataset
valid_all_kirc_samples = methylation_kirc_pd.index.tolist()

#filter the clinical dataset to keep only the samples that are present in the methylation dataset
clinical_kirc_all = clinical_kirc_all[clinical_kirc_all['sample'].isin(valid_all_kirc_samples)]

#this is done to ensure that the clinical and methylation data have the same row size which is needed for splitting the data into training and testing sets

In [ ]:

#split at the PATIENT level (not sample level) to prevent leakage: many patients contribute both a tumor and a matched-normal sample, so a sample-level split can put one in train and the other in holdout
patient_ids = methylation_kirc_pd.index.str[:12]

gss = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=67)
train_idx, holdout_idx = next(gss.split(methylation_kirc_pd, groups=patient_ids))

methylation_kirc_train = methylation_kirc_pd.iloc[train_idx]
methylation_kirc_holdout = methylation_kirc_pd.iloc[holdout_idx]


In [ ]:
#verify the patient-grouped split has zero patient overlap and check class balance
train_patients = set(methylation_kirc_train.index.str[:12])
holdout_patients = set(methylation_kirc_holdout.index.str[:12])
print(f"Patient overlap between train and holdout: {len(train_patients & holdout_patients)}")

train_balance = clinical_kirc_all[clinical_kirc_all['sample'].isin(methylation_kirc_train.index)]['tissue_type.samples'].value_counts()
holdout_balance = clinical_kirc_all[clinical_kirc_all['sample'].isin(methylation_kirc_holdout.index)]['tissue_type.samples'].value_counts()
print(f"Train tissue type distribution:\n{train_balance}")
print(f"Holdout tissue type distribution:\n{holdout_balance}")


Patient overlap between train and holdout: 0
Train tissue type distribution:
tissue_type.samples
Tumor     193
Normal     92
Name: count, dtype: int64
Holdout tissue type distribution:
tissue_type.samples
Tumor     130
Normal     68
Name: count, dtype: int64


In [31]:
print("This is what the training methylation data looks like:")
print(methylation_kirc_train)

This is what the training methylation data looks like:
Composite Element REF  cg00000108  cg00000236  cg00000292  cg00000321  \
TCGA-B0-5108-01A         0.972311    0.909915    0.601544    0.283628   
TCGA-B0-5703-01A         0.964033    0.874732    0.590178    0.454810   
TCGA-CJ-4905-11A         0.960496    0.907699    0.489555    0.437536   
TCGA-B0-4714-01A         0.959082    0.892050    0.539843    0.709672   
TCGA-B2-5635-01A         0.971277    0.933855    0.387262    0.299354   
...                           ...         ...         ...         ...   
TCGA-BP-4993-01A         0.965083    0.926505    0.766406    0.518425   
TCGA-CZ-4865-01A         0.969403    0.915849    0.515882    0.515539   
TCGA-BP-4782-11A         0.959676    0.936125    0.496571    0.396565   
TCGA-B0-4945-11A         0.970876    0.924377    0.491113    0.455013   
TCGA-B4-5843-01A         0.969888    0.948324    0.262626    0.540755   

Composite Element REF  cg00000363  cg00000622  cg00000658  cg0000071

In [32]:
print("\n This is what the holdout methylation data looks like:")
print(methylation_kirc_holdout)


 This is what the holdout methylation data looks like:
Composite Element REF  cg00000108  cg00000236  cg00000292  cg00000321  \
TCGA-B4-5834-01A         0.972071    0.917789    0.426973    0.270070   
TCGA-DV-5575-01A         0.964997    0.931517    0.492524    0.310941   
TCGA-BP-5195-11A         0.972698    0.888925    0.512414    0.574735   
TCGA-BP-5195-01A         0.969650    0.927199    0.461449    0.447459   
TCGA-B8-5165-01A         0.967205    0.916345    0.550989    0.284367   
...                           ...         ...         ...         ...   
TCGA-B0-4813-11A         0.956692    0.906532    0.493675    0.550481   
TCGA-BP-5010-11A         0.967596    0.910573    0.564515    0.533479   
TCGA-B0-4823-11A         0.967950    0.911225    0.522083    0.512706   
TCGA-A3-A6NI-01A         0.955545    0.910482    0.339500    0.437208   
TCGA-CJ-5689-01A         0.975825    0.903064    0.672159    0.541982   

Composite Element REF  cg00000363  cg00000622  cg00000658  cg000007

In [33]:

print("Check the number of NaN values in the training and holdout methylation datasets before imputation")
print(f"Total NaNs remaining in training data: {methylation_kirc_train.isna().sum().sum()}")
print(f"Total NaNs remaining in holdout data: {methylation_kirc_holdout.isna().sum().sum()}")

Check the number of NaN values in the training and holdout methylation datasets before imputation
Total NaNs remaining in training data: 41058
Total NaNs remaining in holdout data: 38463


In [34]:
#impute the methylation training data using median imputation
imputer = SimpleImputer(strategy='median')

#fit and transform the data (note: this returns a numpy array)
methylation_imputed_array = imputer.fit_transform(methylation_kirc_train)

#convert back to a pandas dataframe to keep patient IDs and probe IDs safe
methylation_kirc_train = pd.DataFrame(
    methylation_imputed_array,
    index=methylation_kirc_train.index,      # Patient IDs
    columns=methylation_kirc_train.columns   # CpG Probe IDs
)

print("Rechecking the number of NaN values in training data after median imputation")
print(f"Total NaNs remaining: {methylation_kirc_train.isna().sum().sum()}")

Rechecking the number of NaN values in training data after median imputation
Total NaNs remaining: 0


In [35]:
#perfrom the same imputation on the methylation holdout data
imputer = SimpleImputer(strategy='median')
methylation_imputed_array = imputer.fit_transform(methylation_kirc_holdout)

#convert back to a pandas dataframe to keep patient IDs and probe IDs safe
methylation_kirc_holdout = pd.DataFrame(
    methylation_imputed_array,
    index=methylation_kirc_holdout.index,      # Patient IDs
    columns=methylation_kirc_holdout.columns   # CpG Probe IDs
)

print("Rechecking the number of NaN values in holdout data after median imputation")
print(f"Total NaNs remaining: {methylation_kirc_holdout.isna().sum().sum()}")

Rechecking the number of NaN values in holdout data after median imputation
Total NaNs remaining: 0


In [36]:
#extract the sample names from the methylation dataset to filter the clinical dataset to keep only the samples that are present in the methylation dataset
valid_all_samples = methylation_kirc_train.index.tolist()
valid_all_samples_holdout = methylation_kirc_holdout.index.tolist() # Assuming you have this too

In [37]:

#filter the clinical train dataset to keep only the samples that are present in the methylation dataset
clinical_kirc_train = clinical_kirc_all[clinical_kirc_all['sample'].isin(valid_all_samples)]

#filter the clinical holdout dataset to keep only the samples that are present in the methylation dataset
clinical_kirc_holdout = clinical_kirc_all[clinical_kirc_all['sample'].isin(valid_all_samples_holdout)]

In [38]:
print("This is what the training methylation data and clinical data looks like:")
print(methylation_kirc_train)
print("----------------------------------------------")
print(clinical_kirc_train)

This is what the training methylation data and clinical data looks like:
Composite Element REF  cg00000108  cg00000236  cg00000292  cg00000321  \
TCGA-B0-5108-01A         0.972311    0.909915    0.601544    0.283628   
TCGA-B0-5703-01A         0.964033    0.874732    0.590178    0.454810   
TCGA-CJ-4905-11A         0.960496    0.907699    0.489555    0.437536   
TCGA-B0-4714-01A         0.959082    0.892050    0.539843    0.709672   
TCGA-B2-5635-01A         0.971277    0.933855    0.387262    0.299354   
...                           ...         ...         ...         ...   
TCGA-BP-4993-01A         0.965083    0.926505    0.766406    0.518425   
TCGA-CZ-4865-01A         0.969403    0.915849    0.515882    0.515539   
TCGA-BP-4782-11A         0.959676    0.936125    0.496571    0.396565   
TCGA-B0-4945-11A         0.970876    0.924377    0.491113    0.455013   
TCGA-B4-5843-01A         0.969888    0.948324    0.262626    0.540755   

Composite Element REF  cg00000363  cg00000622  cg0

In [39]:
print("This is what the holdout methylation data and clinical data looks like:")
methylation_kirc_holdout


This is what the holdout methylation data and clinical data looks like:


Composite Element REF,cg00000108,cg00000236,cg00000292,cg00000321,cg00000363,cg00000622,cg00000658,cg00000714,cg00000721,cg00000734,...,ctl_69667417,ctl_70664314,ctl_70700334,ctl_71670310,ctl_71678368,ctl_71718498,ctl_73635489,ctl_73784382,ctl_73794434,ctl_74666473
TCGA-B4-5834-01A,0.972071,0.917789,0.426973,0.270070,0.351346,0.011284,0.862835,0.141163,0.965383,0.074103,...,0.889554,0.100888,0.068487,0.951324,0.048633,0.249223,0.052203,0.951095,0.959078,0.961796
TCGA-DV-5575-01A,0.964997,0.931517,0.492524,0.310941,0.226339,0.012376,0.874163,0.160123,0.960429,0.067214,...,0.925697,0.037106,0.071529,0.960663,0.044961,0.147622,0.050874,0.937494,0.958510,0.949927
TCGA-BP-5195-11A,0.972698,0.888925,0.512414,0.574735,0.201092,0.024069,0.912388,0.196851,0.956501,0.080242,...,0.946776,0.027305,0.059446,0.964031,0.060363,0.179446,0.062773,0.946509,0.943428,0.962431
TCGA-BP-5195-01A,0.969650,0.927199,0.461449,0.447459,0.224090,0.008947,0.954651,0.139885,0.960948,0.077233,...,0.943960,0.018929,0.062069,0.967989,0.047160,0.172492,0.048383,0.957683,0.951249,0.963472
TCGA-B8-5165-01A,0.967205,0.916345,0.550989,0.284367,0.180671,0.008042,0.877757,0.124144,0.959560,0.048545,...,0.940954,0.029113,0.073797,0.968861,0.043066,0.169768,0.045810,0.957088,0.956074,0.949140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-B0-4813-11A,0.956692,0.906532,0.493675,0.550481,0.114842,0.010328,0.848741,0.187480,0.935683,0.056698,...,0.894490,0.013092,0.086768,0.939911,0.109435,0.202068,0.096711,0.910936,0.899721,0.951262
TCGA-BP-5010-11A,0.967596,0.910573,0.564515,0.533479,0.155655,0.009449,0.844393,0.163511,0.955114,0.057939,...,0.944623,0.050170,0.055988,0.967544,0.049648,0.170393,0.053387,0.957767,0.960849,0.950621
TCGA-B0-4823-11A,0.967950,0.911225,0.522083,0.512706,0.230194,0.009638,0.822798,0.171498,0.950986,0.055662,...,0.930867,0.030663,0.066078,0.965939,0.050796,0.222552,0.061422,0.950035,0.951596,0.953344
TCGA-A3-A6NI-01A,0.955545,0.910482,0.339500,0.437208,0.189256,0.012322,0.874743,0.120242,0.920495,0.068927,...,0.937918,0.022148,0.091393,0.967872,0.056139,0.125359,0.055387,0.959845,0.967979,0.973819


In [40]:
print("This is what the holdout methylation data and clinical data looks like:")
clinical_kirc_holdout

This is what the holdout methylation data and clinical data looks like:


,sample,tissue_type.samples
0,TCGA-B0-5695-01A,Tumor
24,TCGA-B0-4712-11A,Normal
25,TCGA-B0-4712-01A,Tumor
43,TCGA-CZ-4856-01A,Tumor
44,TCGA-CZ-4856-11A,Normal
...,...,...
931,TCGA-B2-3924-01B,Tumor
944,TCGA-B0-5402-01A,Tumor
945,TCGA-B0-5402-11A,Normal
948,TCGA-CZ-4866-01A,Tumor


In [41]:
#save the clinical dataset for later use
clinical_kirc_train.to_csv('../data/processed/clinical_kirc_train.csv')
clinical_kirc_holdout.to_csv('../data/processed/clinical_kirc_holdout.csv')

In [42]:
#save the methylation datasets for later use; use parquet for faster read/write and smaller file size
methylation_kirc_train.to_parquet('../data/processed/methylation_kirc_train.parquet')
methylation_kirc_holdout.to_parquet('../data/processed/methylation_kirc_holdout.parquet')

In [43]:
#for installing parquet
#!pip3 install fastparquet